# S&P 500 Market Map Pipeline

Este notebook replica el flujo completo de `market-map` de forma simple.

Orden:
1. Extraer la lista actual del S&P 500 desde Wikipedia
2. Bajar precios históricos diarios
3. Calcular market caps y pesos actuales
4. Calcular retornos por empresa
5. Construir los JSON finales que usa la app


In [ ]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path('/Users/sbc/projects/Plots/sp500/market-map')
SCRIPTS = PROJECT_ROOT / 'data' / 'scripts'
DATA_ROOT = PROJECT_ROOT / 'data'
PUBLIC_DATA = PROJECT_ROOT / 'public' / 'data'

PROJECT_ROOT, SCRIPTS, DATA_ROOT, PUBLIC_DATA


## Helper

Esta función ejecuta comandos desde la carpeta `market-map` y falla de inmediato si algo sale mal.


In [ ]:
def run(cmd):
    print('>>', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f'Command failed: {cmd}')
    return result


## Step 1. Wikipedia constituents

Baja la página de Wikipedia y guarda dos archivos:
- `constituents.html`: snapshot crudo
- `constituents.json`: lista limpia de empresas, ticker, sector e industria


In [ ]:
run(['python3', str(SCRIPTS / '01-extract-wikipedia.py')])


## Step 2. Historical prices

Usa la lista actual del índice para bajar precios ajustados diarios con `yfinance`.

Salida principal:
- `sp500_timeseries_15y_wide.csv`
- `sp500_timeseries_15y_long.csv`
- `sp500_timeseries_15y_meta.json`


In [ ]:
run(['python3', str(SCRIPTS / '02-build-timeseries-csv.py')])


## Step 3. Company weights

Calcula market caps y pesos actuales de las empresas del índice.

Salida principal:
- `sp500_companies_wiki_yfinance.csv`


In [ ]:
run(['python3', str(SCRIPTS / '03-build-company-weights-csv.py')])


## Step 4. Company returns

Calcula retornos por empresa para los horizontes usados por el market map:
- `1D`
- `YTD`
- `1Y`
- `5Y`
- `10Y`

Salida principal:
- `sp500_market_map_returns.csv`


In [ ]:
run(['python3', str(SCRIPTS / '04-build-returns-csv.py')])


## Step 5. Final web datasets

Convierte los CSV anteriores en los JSON finales que consume la app React.

Salida final:
- `public/data/sp500-market-map.json`
- `public/data/sp500-sector-returns.json`


In [ ]:
run(['python3', str(SCRIPTS / '05-build-market-map-data.py')])


## One-command alternative

Si no quieres correr paso por paso, este script hace todo el pipeline completo:


In [ ]:
run(['python3', str(SCRIPTS / '00-update-sources.py')])


## Quick checks

Estas celdas ayudan a verificar que los archivos principales quedaron actualizados.


In [ ]:
for path in [
    DATA_ROOT / 'extracted' / 'wikipedia' / 'constituents.json',
    DATA_ROOT / 'manual' / 'notebook' / 'sp500_timeseries_15y_wide.csv',
    DATA_ROOT / 'manual' / 'notebook' / 'sp500_companies_wiki_yfinance.csv',
    DATA_ROOT / 'manual' / 'notebook' / 'sp500_market_map_returns.csv',
    PUBLIC_DATA / 'sp500-market-map.json',
    PUBLIC_DATA / 'sp500-sector-returns.json',
]:
    print(path, '->', 'OK' if path.exists() else 'MISSING')
